In [ ]:
# ============================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ============================================================

!apt-get update -qq && apt-get install -y ffmpeg build-essential sox libsox-dev
!pip install -q librosa soundfile numpy scipy pandas tqdm
!pip install -q praat-parselmouth pyworld torchcrepe resampy
!pip install -q demucs

print("✓ All dependencies installed")

In [ ]:
# ============================================================
# CELL 2 — IMPORTS AND CONFIGURATION
# ============================================================

import os
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from tqdm import tqdm
import subprocess
import shutil

# Configure the two language datasets.
# Change only the input paths to the locations of your recordings.
LANGUAGES = {
    "hindi": {
        "input": "/content/drive/MyDrive/five_songs_dataset/hindi",
        "fft_output": "/content/drive/MyDrive/five_language_output/hindi/fft"
    },

    "bengali": {
        "input": "/content/drive/MyDrive/five_songs_dataset/bengali",
        "fft_output": "/content/drive/MyDrive/five_language_output/bengali/fft"
    }
}

print("✓ Configuration loaded")
print("Languages:", list(LANGUAGES.keys()))

In [ ]:
# ============================================================
# CELL 3 — CREATE OUTPUT DIRECTORIES
# ============================================================

for lang, paths in LANGUAGES.items():

    os.makedirs(paths["fft_output"], exist_ok=True)

    # Store separated vocals and reference samples separately.
    os.makedirs(
        f"{paths['fft_output']}/vocals",
        exist_ok=True
    )

    os.makedirs(
        f"{paths['fft_output']}/references",
        exist_ok=True
    )

    print(f"✓ Created directories for {lang}")

print("\n✓ All directory structures created")

In [ ]:
# ============================================================
# CELL 4 — LOAD ALL AUDIO FILES
# ============================================================

def load_audio_files(input_dir):
    """
    Find all supported audio recordings recursively.
    """

    audio_files = []

    for root, dirs, files in os.walk(input_dir):

        for file in files:

            if file.lower().endswith(
                (".mp3", ".wav", ".flac", ".m4a", ".ogg")
            ):
                audio_files.append(
                    os.path.join(root, file)
                )

    return sorted(audio_files)


# Load Hindi and Bengali recordings.
dataset_info = {}

for lang, paths in LANGUAGES.items():

    files = load_audio_files(paths["input"])

    dataset_info[lang] = files

    print(
        f"{lang.capitalize()}: "
        f"{len(files)} songs found"
    )

total_songs = sum(
    len(files)
    for files in dataset_info.values()
)

print(
    f"\n✓ Total songs across all languages: "
    f"{total_songs}"
)

In [ ]:
# ============================================================
# CELL 5 — VOCAL SEPARATION USING DEMUCS
# ============================================================

def separate_vocals_demucs(
    audio_path,
    output_dir,
    lang_name
):
    """
    Separate vocals from the original song using Demucs.

    The extracted vocal track is used as the source audio
    for the subsequent FFT-based voice conversion.
    """

    try:

        basename = os.path.splitext(
            os.path.basename(audio_path)
        )[0]

        temp_dir = f"/content/demucs_temp/{lang_name}"

        os.makedirs(
            temp_dir,
            exist_ok=True
        )

        # Run Demucs using the htdemucs model and retain
        # only the vocal stem.
        cmd = (
            f"python -m demucs.separate "
            f"-n htdemucs "
            f"--two-stems=vocals "
            f"-o {temp_dir} "
            f"'{audio_path}'"
        )

        result = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            timeout=300
        )

        if result.returncode == 0:

            vocal_source = (
                f"{temp_dir}/htdemucs/"
                f"{basename}/vocals.wav"
            )

            if os.path.exists(vocal_source):

                vocal_dest = os.path.join(
                    output_dir,
                    f"{basename}_vocals.wav"
                )

                shutil.copy(
                    vocal_source,
                    vocal_dest
                )

                # Remove temporary Demucs files after
                # successfully copying the vocal track.
                shutil.rmtree(
                    f"{temp_dir}/htdemucs/{basename}",
                    ignore_errors=True
                )

                return vocal_dest

        return None

    except Exception as e:

        print(
            f"  Error separating "
            f"{os.path.basename(audio_path)}: {e}"
        )

        return None

In [ ]:
# ============================================================
# CELL 6 — PROCESS VOCAL SEPARATION
# ============================================================

separated_vocals = {}

for lang, files in dataset_info.items():

    print("\n" + "=" * 60)
    print(f"🎵 SEPARATING {lang.upper()} VOCALS")
    print("=" * 60)

    separated_vocals[lang] = []

    output_dir = LANGUAGES[lang]["fft_output"]

    for audio_path in tqdm(
        files,
        desc=f"Demucs {lang}"
    ):

        vocal_path = separate_vocals_demucs(
            audio_path,
            f"{output_dir}/vocals",
            lang
        )

        if vocal_path is not None:

            separated_vocals[lang].append({
                "name": os.path.basename(audio_path),
                "original": audio_path,
                "vocals": vocal_path
            })

    print(
        f"✓ {lang.capitalize()}: "
        f"{len(separated_vocals[lang])}/"
        f"{len(files)} vocals extracted"
    )

In [ ]:
# ============================================================
# CELL 7 — CREATE LANGUAGE-SPECIFIC REFERENCE SAMPLES
# ============================================================

def create_reference_sample(
    audio_path,
    output_path,
    duration=15
):
    """
    Extract a 15-second reference segment from the
    middle portion of a separated vocal recording.
    """

    try:

        # Resample every recording to 44.1 kHz to maintain
        # the same processing rate used by the FFT pipeline.
        audio, sr = librosa.load(
            audio_path,
            sr=44100
        )

        total_samples = len(audio)

        # Select a position approximately one-third into
        # the recording.
        start_pos = total_samples // 3

        sample_length = sr * duration

        if (
            start_pos + sample_length
            <= total_samples
        ):

            sample = audio[
                start_pos:
                start_pos + sample_length
            ]

        else:

            sample = (
                audio[-sample_length:]
                if total_samples >= sample_length
                else audio
            )

        # Normalize the reference before spectral analysis.
        sample = librosa.util.normalize(sample)

        sf.write(
            output_path,
            sample,
            44100
        )

        return True

    except Exception as e:

        print(
            f"Error creating reference: {e}"
        )

        return False


# Create one reference sample per language.
reference_samples = {}

for lang, vocals in separated_vocals.items():

    if len(vocals) == 0:
        continue

    reference_path = (
        f"{LANGUAGES[lang]['fft_output']}/"
        f"references/{lang}_reference.wav"
    )

    success = create_reference_sample(
        vocals[0]["vocals"],
        reference_path
    )

    if success:

        reference_samples[lang] = reference_path

        print(
            f"✓ {lang.capitalize()}: "
            f"Reference created"
        )

print("\n✓ All reference samples created")

In [ ]:
# ============================================================
# CELL 8 — FFT-BASED VOICE CONVERSION
# ============================================================

def fft_voice_conversion(
    source_path,
    reference_path,
    output_path,
    strength=0.7
):
    """
    FFT-based spectral envelope transfer.

    Source:
        Provides the original vocal signal and phase.

    Reference:
        Provides the spectral magnitude characteristics
        used for conversion.

    strength:
        Controls the amount of reference spectral influence.
    """

    try:

        # Load both source and reference at 44.1 kHz.
        source, sr = librosa.load(
            source_path,
            sr=44100
        )

        reference, sr_ref = librosa.load(
            reference_path,
            sr=44100
        )

        # Skip recordings that are too short for processing.
        if (
            len(source) < sr
            or len(reference) < sr
        ):
            return False

        # FFT parameters used by the original pipeline.
        n_fft = 2048
        hop_length = 512

        # Convert both signals into the frequency domain.
        source_stft = librosa.stft(
            source,
            n_fft=n_fft,
            hop_length=hop_length
        )

        reference_stft = librosa.stft(
            reference,
            n_fft=n_fft,
            hop_length=hop_length
        )

        # Extract source magnitude and preserve
        # the original source phase.
        source_mag = np.abs(source_stft)
        source_phase = np.angle(source_stft)

        # Extract the reference spectral magnitude.
        reference_mag = np.abs(reference_stft)

        # Repeat the reference spectrum when the reference
        # is shorter than the source.
        if (
            reference_mag.shape[1]
            < source_mag.shape[1]
        ):

            repeat_times = (
                source_mag.shape[1]
                // reference_mag.shape[1]
            ) + 1

            reference_mag = np.tile(
                reference_mag,
                (1, repeat_times)
            )

        # Match reference length to source length.
        reference_mag = reference_mag[
            :,
            :source_mag.shape[1]
        ]

        # Blend source and reference spectral envelopes.
        converted_mag = (
            source_mag * (1 - strength)
            + reference_mag * strength
        )

        # Reconstruct the complex spectrum using
        # the original source phase.
        converted_stft = (
            converted_mag
            * np.exp(1j * source_phase)
        )

        # Convert the modified spectrum back to audio.
        converted_audio = librosa.istft(
            converted_stft,
            hop_length=hop_length
        )

        # Normalize the generated audio.
        converted_audio = (
            librosa.util.normalize(
                converted_audio
            )
        )

        # Remove low-level silence from the beginning
        # and end of the generated recording.
        converted_audio, _ = librosa.effects.trim(
            converted_audio,
            top_db=20
        )

        # Save as 16-bit PCM WAV.
        sf.write(
            output_path,
            converted_audio,
            44100,
            subtype="PCM_16"
        )

        return True

    except Exception as e:

        print(
            f"FFT conversion error: {e}"
        )

        return False


print("✓ FFT voice conversion function defined")

In [ ]:
# ============================================================
# CELL 9 — GENERATE SYNTHETIC SINGING VOICE SAMPLES
# ============================================================

print("=" * 70)
print("STEP 2: FFT-BASED VOICE CONVERSION")
print("=" * 70)

fft_results = {}

for lang, vocals in separated_vocals.items():

    print(f"\n{'=' * 70}")
    print(
        f"🎵 PROCESSING {lang.upper()} WITH FFT"
    )
    print(f"{'=' * 70}")

    if lang not in reference_samples:

        print(
            f"⚠️ No reference available for "
            f"{lang}, skipping..."
        )

        continue

    fft_results[lang] = []

    reference_path = reference_samples[lang]

    # The first vocal recording is used to create the
    # language-specific reference, so conversion starts
    # from the remaining recordings.
    for i, vocal_data in enumerate(
        tqdm(
            vocals[1:],
            desc=f"FFT {lang}"
        )
    ):

        output_name = (
            f"fft_{lang}_{i + 1:03d}.wav"
        )

        output_path = os.path.join(
            LANGUAGES[lang]["fft_output"],
            output_name
        )

        success = fft_voice_conversion(
            vocal_data["vocals"],
            reference_path,
            output_path,
            strength=0.65
        )

        if success:

            fft_results[lang].append({
                "output": output_path,
                "original": vocal_data["name"]
            })

            print(
                f"✓ Saved {output_name}"
            )

print("\n" + "=" * 70)
print("✓ FFT CONVERSION COMPLETED")
print("=" * 70)

for lang, results in fft_results.items():

    print(
        f"  {lang.capitalize()}: "
        f"{len(results)} songs converted"
    )